# Week 8 — End-to-End Business Analysis: Guided Practice
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

In the demo you watched three full analyses being built. Watching is not the same as writing, and on
Thursday you present work that is marked on SQL you wrote yourself. This notebook is where you write it.

It has two parts, and they work differently:

**Part A — Build the three analyses yourself (Exercises 1–3).** These mirror the demo's Analysis 1, 2
and 3, but the query cells are empty. Each returns **many rows**, so there is no single right answer a
computer can tick — instead each question lists the verified anchor figures your output must line up
with, and you check your own work against them. That is exactly the judgement your presentation is
marked on.

**Part B — Verify your baseline (Exercises 4–5).** Two queries with locked, curriculum-verified answers.
These are your safety net: run them before every presentation rehearsal, and before you trust any number
a query — or DeepSeek — hands you.

**How the cells work.** A question in Part B comes as three cells: the question, a blank `%%sql qN <<`
answer cell, and a check cell that prints ✅ when your query is right. Part A questions are two cells:
the question and a blank `%%sql` cell whose result displays straight below it.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom.

### Setup — run this cell first

It loads all 8 Olist tables into a file-based SQLite database and connects the `%%sql` magic to it.
Nothing below will run until this has finished. Because `autopandas` is on, every `%%sql` result comes
back as a pandas DataFrame, which is what lets the check cells in Part B inspect your answer.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


---
# Part A — Build the three analyses

Rules that apply to every question in this part, because they are the rules your Thursday work is
marked under:

1. **SQL only.** No pandas beyond displaying the result.
2. **Use a `WITH` block** whenever the query needs more than one step. Query readability carries 10 of
   the 40 marks, and a CTE is the cheapest way to earn them.
3. **Watch the grain.** `order_items`, `order_payments` and `order_reviews` each hold *many rows per
   order*. Join two of them directly and every `SUM`, `AVG` and `COUNT(*)` inflates — silently, with no
   error. If your question touches two of them, collapse one to one row per `order_id` in a CTE first.
4. **Check your number against an anchor** before you believe it. Each question below gives you the
   anchors to check against.

## Exercise 1 — Platform growth: 2017 vs 2018

Leadership's first question is *how has the platform grown?* Write one query that answers it properly,
by reporting growth on four faces at once — one row per year, two rows in total.

For each of **2017** and **2018**, return:

| Column | What it holds |
|---|---|
| `year` | the four-digit year, pulled from `orders.order_purchase_timestamp` |
| `total_orders` | how many distinct orders were placed |
| `total_revenue` | the total `payment_value` taken, rounded to 2 dp |
| `avg_order_value` | the average `payment_value`, rounded to 2 dp |
| `unique_customers` | how many distinct customers placed them |

Sort by `year`.

**How to build it.** Shape the yearly figures inside a `WITH` block, then `SELECT` from it — that is the
Week 7 pattern, and it keeps the query readable. `order_purchase_timestamp` is **TEXT**, not a date, so
the year comes from `strftime('%Y', ...)` (Week 4); grouping by that expression's alias is what gives you
one row per year. Payment values live in `order_payments`, so you need a `JOIN` (Week 3) — and because a
single order can be settled in several payment rows, count orders with `COUNT(DISTINCT order_id)` (Week 2)
or you will overcount.

**Restrict the `WHERE` clause to 2017 and 2018.** The dataset also holds **329 orders from 2016** — a
partial pilot year. Comparing a full year against it would produce a growth rate that is spectacular and
meaningless. Excluding a year is a judgement call, so be ready to say out loud why you made it.

**Anchors — check your output against these:**
- `total_orders` should read **45,101** for 2017 and **54,011** for 2018.
- Two rows, no more. If you see a 2016 row, your `WHERE` clause is not doing its job.

**Then answer in one sentence:** did revenue grow faster or slower than order count — and what does that
say about the average basket?

In [ ]:
%%sql
-- Exercise 1 — platform growth, one row per year (2017 and 2018).
-- Your query here
--
-- Sketch before you write:
--   WITH yearly_summary AS ( ... one row per year ... )
--   SELECT ... FROM yearly_summary ORDER BY year

## Exercise 2 — Delivery performance by state

Before you can link delivery to satisfaction, you have to measure delivery. Write one query that reports,
for each `customer_state`, the **ten fastest states** by average delivery time:

| Column | What it holds |
|---|---|
| `customer_state` | the two-letter state code |
| `delivered_orders` | how many delivered orders that state received |
| `avg_days` | average days from purchase to delivery, rounded to 1 dp |
| `late_count` | how many arrived after the estimated delivery date |
| `late_pct` | `late_count` as a percentage of `delivered_orders`, rounded to 1 dp |
| `national_avg_days` | the average of `avg_days` across **all** states, rounded to 1 dp |

Keep only rows where `order_status = 'delivered'` **and** `order_delivered_customer_date IS NOT NULL`,
sort by `avg_days` ascending, and `LIMIT 10`.

**Three things to get right.** SQLite has no date type and no `DATEDIFF`, so a duration in days is
`julianday(delivered) - julianday(purchased)`. "How many rows satisfy a condition, alongside other
aggregates" is `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` — here the condition for *late* is that the
delivered date is greater than the estimated one. And `late_pct` must force REAL division: write
`late_count * 100.0 / delivered_orders`, because `* 100` divides two integers into an integer and hands
you `0` for every single state.

`national_avg_days` is the interesting column. Compute it with a window function over an **empty**
`OVER ()` — no `PARTITION BY`, no `ORDER BY` — so the window is the entire result set and the same
benchmark lands on every row. That is what lets a reader judge each state against the national picture
instead of in isolation.

**Anchors — check your output against these:**
- Exactly 10 rows.
- `national_avg_days` will come out **higher than 12.6**, the platform-wide average delivery time. That
  is not a bug, and explaining it is worth marks: your column averages **state averages**, giving tiny
  Roraima the same weight as São Paulo, while 12.6 averages **orders**, most of which sit in the fast
  southeastern states. Two legitimate questions, two different numbers — say which one you mean.
- Across the whole platform **7,826** deliveries (**8.1%**) arrived late, so a state's `late_pct` should
  be read against roughly that figure.
- Read `avg_days` next to `delivered_orders`. São Paulo alone holds **41,746** customers; a state with a
  few hundred deliveries can top this table on luck.

In [ ]:
%%sql
-- Exercise 2 — delivery performance per customer state, ten fastest first.
-- Your query here
--
-- Sketch before you write:
--   WITH delivery_stats AS ( ... one row per state ... )
--   SELECT ..., ROUND(AVG(avg_days) OVER (), 1) AS national_avg_days
--   FROM delivery_stats ORDER BY avg_days ASC LIMIT 10

## Exercise 3 — Does delivery speed explain review scores?

Now join the two halves of the brief's fourth question. Instead of grouping by state, group by
`review_score` (1 to 5) and describe what delivery looked like for the orders behind each score.

Return one row per score, sorted by `review_score`:

| Column | What it holds |
|---|---|
| `review_score` | 1 to 5 |
| `avg_delivery_days` | average days from purchase to delivery, rounded to 1 dp |
| `order_count` | how many rows sit behind that score |
| `late_pct` | percentage of them that arrived after the estimated date, rounded to 1 dp |

Same filter as Exercise 2: delivered orders with a real delivery date only — an order that never arrived
has no delivery duration to average.

**A grain point worth pausing on.** `order_reviews` holds many rows per order, so a `COUNT(*)` here counts
**reviews**, not orders. This query is still safe because it joins exactly **one** many-per-order table.
The moment you add `order_items` or `order_payments` beside `order_reviews`, rows multiply and every
aggregate inflates — the single most common way a capstone number comes out wrong.

**Anchors — check your output against these:**
- Five rows, one per star rating.
- Platform-wide the average review score is **4.09**, and **57,328** reviews (**57.8%** of all reviews)
  are 5-star. Your 5-star `order_count` will sit a little **below** 57,328 — and you should be able to
  explain the gap without looking it up, because you filtered to delivered orders with a delivery date.

**Then answer in one sentence:** read down `avg_delivery_days`, then down `late_pct`, and describe the
shape. That sentence is your slide. Add the two cautions your group should say out loud: this is
*association*, not proof of cause, and every row here is a delivered order — the customers whose orders
never arrived are excluded entirely, and they are unlikely to be the happy ones.

In [ ]:
%%sql
-- Exercise 3 — delivery experience broken down by the review score it earned.
-- Your query here
--
-- Sketch before you write:
--   WITH delivery_and_review AS ( ... one row per review_score ... )
--   SELECT ... FROM delivery_and_review ORDER BY review_score

---
# Part B — Verify your baseline

The two questions below have locked, verified answers, and they are the most practical cells in this
notebook. Run them **before every rehearsal and before your presentation**. If these numbers come out
right, your database loaded correctly and any figure you build on top of it starts from solid ground. If
they come out wrong, stop — nothing downstream is trustworthy, no matter how plausible it looks.

They are also step 3 of the DeepSeek protocol: when the AI drafts a query for you, make it reproduce a
number you already trust before you believe a number you don't.

Each question is three cells: the question, a blank `%%sql qN <<` answer cell, and a check cell that
prints ✅. **The column aliases matter** — the check cells look the columns up by name, so use exactly
the aliases the question asks for.

## Exercise 4 — Platform overview KPIs

Write a **single query** over the `orders` table that returns one row with four columns:

| Alias | What it holds |
|---|---|
| `total_orders` | the count of distinct `order_id` values |
| `delivered` | how many orders have `order_status = 'delivered'` |
| `canceled` | how many orders have `order_status = 'canceled'` |
| `delivered_pct` | delivered as a percentage of all orders, rounded to 1 dp |

Do it **without a `WHERE` clause** — that is the point of the exercise. `SUM(CASE WHEN ... THEN 1 ELSE 0
END)` counts rows matching a condition while still counting every other row too, which is how you get
three different counts from one pass over the table. For `delivered_pct`, remember to force REAL
division: `1.0` on the `THEN` branch, or `* 100.0`, otherwise integer division returns `0`.

**Expected:** 99,441 total orders · 96,478 delivered · 625 canceled · 97.0% delivered

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert int(q4.iloc[0]['total_orders']) == 99441, "Q4: expected 99,441 total orders"
assert int(q4.iloc[0]['delivered']) == 96478, "Q4: expected 96,478 delivered orders"
assert int(q4.iloc[0]['canceled']) == 625, "Q4: expected 625 canceled orders"
assert round(float(q4.iloc[0]['delivered_pct']), 1) == 97.0, "Q4: expected delivered_pct of 97.0"
print("✅ Q4 correct")
q4  # show the result of your query

## Exercise 5 — Total GMV verification

Gross merchandise value is the headline money figure in your presentation, so it is the one you least want
to get wrong. Write a **single query** over `order_items` returning one row with three columns:

| Alias | What it holds |
|---|---|
| `product_revenue` | the sum of `price`, rounded to 2 dp |
| `freight_revenue` | the sum of `freight_value`, rounded to 2 dp |
| `total_gmv` | the sum of `price + freight_value`, rounded to 2 dp |

**Use `order_items` alone.** It is tempting to join `orders` or `order_payments` "for completeness", and
that is precisely the trap: `order_items` and `order_payments` are both many-per-order, so joining them
repeats every item row once per payment row and inflates the total. No error is raised. The query runs,
the number looks plausible, and it is wrong. The money question only ever needed `order_items`, at item
grain.

**Expected:** product revenue R$13,591,643.70 · freight revenue R$2,251,909.54 · total GMV
R$15,843,553.24

In [ ]:
%%sql q5 <<
-- Your query here

In [ ]:
# --- CHECK Q5 — do not edit ---
assert round(float(q5.iloc[0]['product_revenue']), 2) == 13591643.70, \
    "Q5: expected product_revenue of 13,591,643.70"
assert round(float(q5.iloc[0]['freight_revenue']), 2) == 2251909.54, \
    "Q5: expected freight_revenue of 2,251,909.54"
assert round(float(q5.iloc[0]['total_gmv']), 2) == 15843553.24, \
    "Q5: expected total_gmv of 15,843,553.24"
print("✅ Q5 correct")
q5  # show the result of your query

---
## Before Thursday

Exercises 1–3 are three of the five analyses your group presents. The remaining two — **top 5 product
categories by revenue** and the **seller tier breakdown** (the `CASE WHEN` tiering pattern from Week 7) —
plus the business question your group chose for itself, are yours to build in your own notebook.

Two anchors for that work: the platform's top category by revenue is **health_beauty** at
**R$1,258,681.34**, and the single best-performing seller took **R$229,472.63**. If your category or
seller query reproduces those, its logic is sound.

A last word on the standard. Leadership will not read your SQL — they will read four or five numbers and
one sentence about what each one means. So for every result you present, write the sentence. And before
any number goes on a slide, run Part B above and make sure both ticks are green.